## Multivariate Imputation by Chainded Equations for Missing Value | MICE Algortihm | iterative Imputer

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

#import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

from sklearn.linear_model import LinearRegression

Loaded the CSV, kept only 4 columns

Divided all values by 10,000 to make numbers smaller

np.round rounded them off

seed(9) ensures same 5 random rows are picked every time you run it

In [2]:
url = "https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/main/day40-iterative-imputer/50_Startups.csv"

df = pd.read_csv(url)

np.random.seed(9)
df = df.sample(5)
df



,R&D Spend,Administration,Marketing Spend,State,Profit
21,78389.47,153773.43,299737.29,New York,111313.02
37,44069.95,51283.14,197029.42,California,89949.14
2,153441.51,101145.55,407934.54,Florida,191050.39
14,119943.24,156547.42,256512.92,Florida,132602.65
44,22177.74,154806.14,28334.72,California,65200.33


### Drop the target column

In [3]:
df = df.iloc[:,0:-1]
df

#Removed the Profit column — kept only the 3 feature columns.

,R&D Spend,Administration,Marketing Spend,State
21,78389.47,153773.43,299737.29,New York
37,44069.95,51283.14,197029.42,California
2,153441.51,101145.55,407934.54,Florida
14,119943.24,156547.42,256512.92,Florida
44,22177.74,154806.14,28334.72,California


### Manually insert NaN values


In [4]:
df.iloc[1,0] = np.nan
df.iloc[3,1] = np.nan
df.iloc[-1,-1] = np.nan

## Artificially created 3 missing values — one in each column — to simulate real-world missing data.

In [5]:
df.head()

,R&D Spend,Administration,Marketing Spend,State
21,78389.47,153773.43,299737.29,New York
37,NaN,51283.14,197029.42,California
2,153441.51,101145.55,407934.54,Florida
14,119943.24,NaN,256512.92,Florida
44,22177.74,154806.14,28334.72,NaN


### Iteration 0 — Fill with Column Mean


In [6]:
# Step 1- Impute all missing values with mean of respective col

df0 = pd.DataFrame()

df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())


## This is just the starting point — fill every NaN with the average of its column. Not accurate, but gives regression something to work with.


In [7]:
# 0th Iteration
df0

,R&D Spend,Administration,Marketing Spend
21,78389.47,153773.430,299737.29
37,93487.99,51283.140,197029.42
2,153441.51,101145.550,407934.54
14,119943.24,115252.065,256512.92
44,22177.74,154806.140,28334.72


In [8]:
# Remove the col1 imputed value
df1 = df0.copy()

df1.iloc[1,0] = np.nan  ##  # put nan back (remove the mean we filled

df1

,R&D Spend,Administration,Marketing Spend
21,78389.47,153773.430,299737.29
37,NaN,51283.140,197029.42
2,153441.51,101145.550,407934.54
14,119943.24,115252.065,256512.92
44,22177.74,154806.140,28334.72


In [9]:
# Use first 3 rows to build a model and use the last for prediction

X = df1.iloc[[0,2,3,4],1:3]  ## # Admin + Marketing of other 4 rows
X

,Administration,Marketing Spend
21,153773.430,299737.29
2,101145.550,407934.54
14,115252.065,256512.92
44,154806.140,28334.72


In [10]:
y = df1.iloc[[0,2,3,4],0]  ##  # their R&D values

In [11]:
lr = LinearRegression()
lr.fit(X.values, y)
lr.predict(df1.iloc[1,1:].values.reshape(1, -1))  ## # predict for row 37

array([176269.94792016])

In [12]:
df1.iloc[1,0] = 23.14
## Put NaN back → train model on other 4 rows → predict → fill it in.

In [13]:
df1

,R&D Spend,Administration,Marketing Spend
21,78389.47,153773.430,299737.29
37,23.14,51283.140,197029.42
2,153441.51,101145.550,407934.54
14,119943.24,115252.065,256512.92
44,22177.74,154806.140,28334.72


## Step 2 — Impute Administration (row 14)

In [14]:
# Remove the col2 imputed value

df1.iloc[3,1] = np.nan  ## # put NaN back

df1

,R&D Spend,Administration,Marketing Spend
21,78389.47,153773.43,299737.29
37,23.14,51283.14,197029.42
2,153441.51,101145.55,407934.54
14,119943.24,NaN,256512.92
44,22177.74,154806.14,28334.72


In [15]:
# Use last 3 rows to build a model and use the first for predictions

X = df1.iloc[[0,1,2,4],[0,2]]  ## # R&D + Marketing of other 4 rows
X

,R&D Spend,Marketing Spend
21,78389.47,299737.29
37,23.14,197029.42
2,153441.51,407934.54
44,22177.74,28334.72


In [16]:
y = df1.iloc[[0,1,2,4],1]  ##  # their Administration values


In [17]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[[3]][X.columns])

array([154429.92931134])

In [18]:
df1.iloc[3,1] = 11.06

In [19]:
df1

,R&D Spend,Administration,Marketing Spend
21,78389.47,153773.43,299737.29
37,23.14,51283.14,197029.42
2,153441.51,101145.55,407934.54
14,119943.24,11.06,256512.92
44,22177.74,154806.14,28334.72


### Step 3 — Impute Marketing Spend (row 44)

In [20]:
# Remove the col3 imputed value
df1.iloc[4,-1] = np.nan  ## # put NaN back


df1

,R&D Spend,Administration,Marketing Spend
21,78389.47,153773.43,299737.29
37,23.14,51283.14,197029.42
2,153441.51,101145.55,407934.54
14,119943.24,11.06,256512.92
44,22177.74,154806.14,NaN


In [21]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[0:4,0:2] ##  # R&D + Admin of other 4 rows
X

,R&D Spend,Administration
21,78389.47,153773.43
37,23.14,51283.14
2,153441.51,101145.55
14,119943.24,11.06


In [22]:
y = df1.iloc[0:4,-1]  ## # their Marketing values
y

21    299737.29
37    197029.42
2     407934.54
14    256512.92
Name: Marketing Spend, dtype: float64

In [23]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[[4]][X.columns])

array([265548.00166418])

In [24]:
df1.iloc[4,-1] = 31.56


In [25]:

# After 1st Iteration
df1

,R&D Spend,Administration,Marketing Spend
21,78389.47,153773.43,299737.29
37,23.14,51283.14,197029.42
2,153441.51,101145.55,407934.54
14,119943.24,11.06,256512.92
44,22177.74,154806.14,31.56


In [26]:

# Subtract 0th iteration from 1st iteration

df1 - df0  ##  # change from iteration 0 to 1

,R&D Spend,Administration,Marketing Spend
21,0.00,0.000,0.00
37,-93464.85,0.000,0.00
2,0.00,0.000,0.00
14,0.00,-115241.005,0.00
44,0.00,0.000,-28303.16


In [27]:
df2 = df1.copy()

df2.iloc[1,0] = np.nan

df2

,R&D Spend,Administration,Marketing Spend
21,78389.47,153773.43,299737.29
37,NaN,51283.14,197029.42
2,153441.51,101145.55,407934.54
14,119943.24,11.06,256512.92
44,22177.74,154806.14,31.56


In [28]:
X = df2.iloc[[0,2,3,4],1:3]
y = df2.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[[1], 1:3])

array([95700.66134052])

In [29]:
df2.iloc[1,0] = 23.78

In [30]:

df2.iloc[3,1] = np.nan
X = df2.iloc[[0,1,2,4],[0,2]]
y = df2.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[[3], [0,2]])

array([149528.52685342])

In [31]:
df2.iloc[3,1] = 11.22


In [32]:
df2.iloc[4,-1] = np.nan

df_clean = df2.dropna()

X = df_clean.iloc[:,0:2]
y = df_clean.iloc[:,-1]

lr = LinearRegression()
lr.fit(X,y)

lr.predict(df2.iloc[[4],0:2])

array([265547.56533123])

In [33]:
df2.iloc[4,-1] = 31.56


In [34]:
df2


,R&D Spend,Administration,Marketing Spend
21,78389.47,153773.43,299737.29
37,23.78,51283.14,197029.42
2,153441.51,101145.55,407934.54
14,119943.24,11.22,256512.92
44,22177.74,154806.14,31.56


In [35]:
df2 - df1  ## # change from iteration 1 to 2

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.0
37,0.64,0.00,0.0
2,0.00,0.00,0.0
14,0.00,0.16,0.0
44,0.00,0.00,0.0


In [36]:
df3 = df2.copy()

df3.iloc[1,0] = np.nan

df3

,R&D Spend,Administration,Marketing Spend
21,78389.47,153773.43,299737.29
37,NaN,51283.14,197029.42
2,153441.51,101145.55,407934.54
14,119943.24,11.22,256512.92
44,22177.74,154806.14,31.56


In [37]:

X = df3.iloc[[0,2,3,4],1:3]
y = df3.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[[1],1:3])

array([95700.69225185])

In [38]:

df3.iloc[1,0] = 24.57


In [39]:

df3.iloc[3,1] = np.nan
X = df3.iloc[[0,1,2,4],[0,2]]
y = df3.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[[3], [0,2]])

array([149528.51647506])

In [40]:
df3.iloc[3,1] = 11.37

In [41]:
df3.iloc[4,-1] = np.nan

df_clean = df3.dropna()

X = df_clean.iloc[:,0:2]
y = df_clean.iloc[:,-1]

lr = LinearRegression()
lr.fit(X,y)

lr.predict(df3.iloc[[4],0:2])

array([265547.02207507])

In [42]:
df3.iloc[4,-1] = 45.53


In [43]:
df2.iloc[3,1] = 11.22


In [44]:
df3

,R&D Spend,Administration,Marketing Spend
21,78389.47,153773.43,299737.29
37,24.57,51283.14,197029.42
2,153441.51,101145.55,407934.54
14,119943.24,11.37,256512.92
44,22177.74,154806.14,45.53


In [45]:
df3-df2  ## # change from iteration 2 to 3

#This checks whether the imputed values are stabilizing. 
# When the difference gets close to 0, the algorithm has converged — meaning further iterations won't change the values much.

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,0.79,0.00,0.00
2,0.00,0.00,0.00
14,0.00,0.15,0.00
44,0.00,0.00,13.97


In [46]:
# # Fill NaNs with column mean first, then repeatedly 
# put each NaN back and use a regression model trained on 
# the other rows to predict a better value — keep doing this until the values stop changing significantly.